# Cointegration-Based Statistical Arbitrage Engine
## Kalman Filter Hedge Ratio — Full Walkthrough

This notebook reproduces the full pipeline interactively:
1. Universe construction & cointegration screening
2. OU process fitting & half-life filtering
3. Kalman filter hedge ratio vs rolling OLS
4. Z-score signal generation
5. Vectorized backtest (IS + OOS)
6. Risk overlays: regime detection, position caps
7. Performance visualisation

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Universe & Cointegration Screening

In [ ]:
from universe.sector_map import SECTOR_TICKERS
from universe.cointegration_screen import download_prices, engle_granger_test, johansen_test, compute_halflife

# Show sector composition
for sector, tickers in SECTOR_TICKERS.items():
    print(f"{sector}: {', '.join(tickers[:5])} ... ({len(tickers)} tickers)")

In [ ]:
# Load screened pairs (run run_pipeline.py first to generate this)
import os
pairs_path = '../data/cointegrated_pairs.csv'
if os.path.exists(pairs_path):
    pairs_df = pd.read_csv(pairs_path)
    print(f"Loaded {len(pairs_df)} cointegrated pairs")
    print(pairs_df.head(10).to_string())
else:
    print('Run run_pipeline.py first to generate cointegrated_pairs.csv')

## 2. Deep-Dive: Single Pair Analysis

We examine one pair in detail to illustrate the methodology.

In [ ]:
T1, T2 = 'KO', 'PEP'   # Classic consumer staples cointegrated pair
START, END = '2019-01-01', '2024-01-01'

prices = download_prices([T1, T2], START, END)
s1, s2 = prices[T1], prices[T2]
print(f'Price data: {len(s1)} trading days, {s1.index[0].date()} to {s1.index[-1].date()}')

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
axes[0].plot(s1.index, s1, label=T1, color='steelblue')
axes[0].plot(s2.index, s2, label=T2, color='orange')
axes[0].set_title(f'{T1} vs {T2} — Adjusted Close')
axes[0].legend()

# Normalised
axes[1].plot(s1.index, s1/s1.iloc[0], label=T1+' (norm)', color='steelblue')
axes[1].plot(s2.index, s2/s2.iloc[0], label=T2+' (norm)', color='orange')
axes[1].set_title('Normalised Prices')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cointegration tests
eg = engle_granger_test(s1, s2)
joh = johansen_test(s1, s2)
hl = compute_halflife(s1 - eg['ols_beta'] * s2)

print(f"Engle-Granger: stat={eg['eg_stat']:.4f}, p-value={eg['eg_pval']:.4f}")
print(f"Johansen rank test pass: {joh}")
print(f"OLS hedge ratio β = {eg['ols_beta']:.4f}")
print(f"Half-life of mean reversion: {hl:.1f} days")

## 3. OU Process Fitting

In [ ]:
from models.ou_model import fit_ou_mle

static_spread = s1 - eg['ols_beta'] * s2
ou = fit_ou_mle(static_spread)
print(f'OU Parameters:')
print(f'  κ (mean reversion speed) = {ou.kappa:.4f}')
print(f'  θ (long-run mean)        = {ou.theta:.4f}')
print(f'  σ (volatility)           = {ou.sigma:.4f}')
print(f'  Half-life                = {ou.half_life:.1f} days')

plt.figure(figsize=(13, 4))
plt.plot(static_spread.index, static_spread, color='steelblue', linewidth=0.9)
plt.axhline(ou.theta, color='red', linestyle='--', label=f'θ={ou.theta:.2f}')
plt.title(f'{T1}/{T2} Spread (Static OLS) — OU fit: HL={ou.half_life:.1f}d')
plt.legend()
plt.show()

## 4. Kalman Filter Hedge Ratio

In [ ]:
from models.kalman_hedge import kalman_hedge_ratio, rolling_ols_beta

kalman_beta, kalman_spread = kalman_hedge_ratio(s1, s2)
ols_beta = rolling_ols_beta(s1, s2, window=60)

SPLIT = int(len(s1) * 0.60)
split_date = s1.index[SPLIT]

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Beta comparison
axes[0].plot(ols_beta.index, ols_beta, label='Rolling OLS (60d)', color='orange', alpha=0.85, linewidth=0.9)
axes[0].plot(kalman_beta.index, kalman_beta, label='Kalman Filter', color='steelblue', linewidth=1.0)
axes[0].axvline(split_date, color='red', linestyle='--', label='IS/OOS split')
axes[0].set_title(f'{T1}/{T2} — Kalman vs Rolling OLS Hedge Ratio β')
axes[0].legend()

# Kalman spread
axes[1].plot(kalman_spread.index, kalman_spread, color='steelblue', linewidth=0.8)
axes[1].axhline(0, color='black', linewidth=0.6)
axes[1].axvline(split_date, color='red', linestyle='--', alpha=0.7)
axes[1].set_title('Kalman-filtered Spread')

plt.tight_layout()
plt.show()

## 5. Z-Score Signals

In [ ]:
from signals.zscore_signals import compute_zscore, generate_signals

zscore = compute_zscore(kalman_spread, window=60)
position = generate_signals(zscore, entry_z=1.5, exit_z=0.5, stop_z=3.0)

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(zscore.index, zscore, color='steelblue', linewidth=0.8)
axes[0].axhline(1.5, color='green', linestyle='--', alpha=0.7, label='Entry +1.5σ')
axes[0].axhline(-1.5, color='green', linestyle='--', alpha=0.7, label='Entry -1.5σ')
axes[0].axhline(0.5, color='gray', linestyle=':', alpha=0.6, label='Exit 0.5σ')
axes[0].axhline(-0.5, color='gray', linestyle=':', alpha=0.6)
axes[0].axhline(3.0, color='red', linestyle='--', alpha=0.5, label='Stop 3.0σ')
axes[0].axhline(-3.0, color='red', linestyle='--', alpha=0.5)
axes[0].set_title(f'{T1}/{T2} — Z-Score with Entry/Exit/Stop Bands')
axes[0].legend(fontsize=8)
axes[0].set_ylim(-4, 4)

axes[1].fill_between(position.index, position, 0,
                     where=position > 0, color='green', alpha=0.4, label='Long spread')
axes[1].fill_between(position.index, position, 0,
                     where=position < 0, color='red', alpha=0.4, label='Short spread')
axes[1].set_title('Position')
axes[1].set_yticks([-1, 0, 1])
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Backtest — In-Sample vs Out-of-Sample

In [ ]:
from backtest.engine import run_backtest, split_insample_oos
from risk.regime_filter import check_regime

s1_is, s1_oos = split_insample_oos(s1, 0.60)
s2_is, s2_oos = split_insample_oos(s2, 0.60)

beta_is, spread_is = kalman_hedge_ratio(s1_is, s2_is)
beta_oos, spread_oos = kalman_hedge_ratio(s1_oos, s2_oos)

z_is = compute_zscore(spread_is)
pos_is = generate_signals(z_is)
pos_is = check_regime(s1_is, s2_is, pos_is)

z_oos = compute_zscore(spread_oos)
pos_oos = generate_signals(z_oos)
pos_oos = check_regime(s1_oos, s2_oos, pos_oos)

bt_is = run_backtest(s1_is, s2_is, beta_is, pos_is, T1, T2)
bt_oos = run_backtest(s1_oos, s2_oos, beta_oos, pos_oos, T1, T2)

print('In-Sample Performance:')
print(f'  Sharpe={bt_is.sharpe:.2f}  Sortino={bt_is.sortino:.2f}  MaxDD={bt_is.max_drawdown:.1%}  Hit={bt_is.hit_rate:.1%}')
print('Out-of-Sample Performance:')
print(f'  Sharpe={bt_oos.sharpe:.2f}  Sortino={bt_oos.sortino:.2f}  MaxDD={bt_oos.max_drawdown:.1%}  Hit={bt_oos.hit_rate:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cum_is = (1 + bt_is.pnl).cumprod() - 1
cum_oos = (1 + bt_oos.pnl).cumprod() - 1

axes[0].plot(cum_is.index, cum_is * 100, color='steelblue')
axes[0].axhline(0, color='black', linewidth=0.6)
axes[0].set_title(f'In-Sample Equity Curve  Sharpe={bt_is.sharpe:.2f}')
axes[0].set_ylabel('Cumulative Return (%)')
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

axes[1].plot(cum_oos.index, cum_oos * 100, color='darkorange')
axes[1].axhline(0, color='black', linewidth=0.6)
axes[1].set_title(f'Out-of-Sample Equity Curve  Sharpe={bt_oos.sharpe:.2f}')
axes[1].set_ylabel('Cumulative Return (%)')
axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

plt.suptitle(f'{T1}/{T2} Strategy Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Portfolio Summary

In [ ]:
import os
summary_path = '../outputs/pair_performance_summary.csv'
if os.path.exists(summary_path):
    summary = pd.read_csv(summary_path)
    print('Top 10 pairs by OOS Sharpe:')
    display_cols = ['pair','half_life','IS_sharpe','OOS_sharpe','max_dd','hit_rate','total_trades_oos']
    print(summary[display_cols].head(10).to_string(index=False))
else:
    print('Run run_pipeline.py first to generate pair_performance_summary.csv')

## 8. Regime Detection Demo

In [ ]:
from risk.regime_filter import check_regime

pos_raw = generate_signals(z_oos)
pos_filtered = check_regime(s1_oos, s2_oos, pos_raw)

paused = (pos_raw != pos_filtered).sum()
print(f'Days paused by regime filter: {paused} ({paused/len(pos_raw):.1%} of OOS period)')

bt_filtered = run_backtest(s1_oos, s2_oos, beta_oos, pos_filtered, T1, T2)
print(f'With regime filter: Sharpe={bt_filtered.sharpe:.2f}  MaxDD={bt_filtered.max_drawdown:.1%}')
print(f'Without filter:     Sharpe={bt_oos.sharpe:.2f}  MaxDD={bt_oos.max_drawdown:.1%}')